# Agent Evaluation Benchmark

Run the shared `src.agent.run_agent()` implementation built with LangChain
`create_agent`. The integrated configuration uses **Qwen3.5 4B**, thinking off,
2,048 tokens for agent turns, and 600 tokens for structured audit calls.

Install dependencies from the repository root, select that Python environment,
and restart the kernel after updating code or configuration:

```bash
python -m pip install -r requirements.txt
```

Set `GENERATION_MODEL=qwen3.5:4b` in `.env`. Follow the [README](../README.md) to
build or verify the Chroma index. The setup cells below start Ollama and download
configured models if missing. Run cells in order.

The model chooses tools and returns judgment fields. Python keeps the original
requirement and retrieved clauses, verifies the selected clause ID, and attaches
the original source text as evidence. Each requirement has its own state, and
tools execute sequentially. A completed Partial or Gap also requires gap logging.
Final model prose cannot replace the validated assessment. See the
[agent walkthrough](../docs/agent.md).

Execution outputs are cleared. Running this notebook makes live model calls;
results stay in memory and no evaluation artifacts are saved.

In [1]:
from importlib.metadata import version

from src.config import BENCHMARK_PATH, GENERATION_MODEL
from src.agent import MAX_RETRIEVAL_K, run_agent
from src.vector_store import load_vector_store
from src.rag import create_audit_client, retrieve_clauses
from src.evaluation import load_benchmark, evaluate
from src.ollama_server import setup_ollama

for package in ("langchain", "langchain-core", "langchain-ollama", "langgraph"):
    print(f"{package}: {version(package)}")

langchain: 1.5.2
langchain-core: 1.8.5
langchain-ollama: 1.1.0
langgraph: 1.2.24


## Configure the Run

Start with five requirements; set `limit=None` for the full 50-requirement benchmark.
These defaults match the integrated agent configuration. Set `verbose=True` for
LangChain graph debug output, which can be lengthy and is not token streaming.
Thinking and output-budget overrides below apply only to agent turns; the audit
client retains its configured settings.

In [2]:
limit = 5
retrievals = 5  # Supported agent range: 1–10.
max_steps = 8
verbose = False
reasoning = False
agent_num_predict = 2048

if limit is not None and (type(limit) is not int or limit < 1):
    raise ValueError("limit must be a positive integer or None")
if type(retrievals) is not int or not 1 <= retrievals <= MAX_RETRIEVAL_K:
    raise ValueError(f"retrievals must be an integer between 1 and {MAX_RETRIEVAL_K}")

print(f"Evaluation mode: Agent (create_agent) | Model: {GENERATION_MODEL} | Retrieval top-K: {retrievals}")
print(f"max_steps={max_steps}, reasoning={reasoning}, agent_num_predict={agent_num_predict}")

Evaluation mode: Agent (create_agent) | Model: qwen3.5:4b | Retrieval top-K: 5
max_steps=8, reasoning=False, agent_num_predict=2048


## Load Ollama and Vector Store

In [3]:
setup_ollama()

Ollama server is already running.
nomic-embed-text is already installed.
qwen3.5:4b is already installed.
Required Ollama models are ready.


In [4]:
vector_store = load_vector_store()
client = create_audit_client()
print(f"Audit model: {client.model} | Audit output budget: {client.num_predict} | Thinking: {client.reasoning}")

Audit model: qwen3.5:4b | Audit output budget: 600 | Thinking: False


## Load Benchmark

Gold labels, expected clause IDs, and reference rationales stay in `benchmark`
for scoring. Only the ID/text pairs in `requirements` are passed to the agent.

In [5]:
benchmark, requirements = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(requirements)} requirements")
display(benchmark.head())
display(benchmark["label"].value_counts())

Loaded 50 requirements


,requirement_id,swe_id,requirement_text,clause_id,label,rationale
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone..."
1,2,SWE-051,The software requirements shall be derived fro...,4.1.3,Partial,Requirements analysis traces to systems engine...
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,..."
3,4,SWE-055,The system shall undergo requirements validati...,4.1.7,Meets,Validation activity confirms the software will...
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi..."


label
Meets      22
Partial    14
Gap        14
Name: count, dtype: int64

## Audit and Evaluate Benchmark

`run_agent()` creates fresh session state for each requirement while sharing the
loaded vector store and model client. The workflow is retrieval, a validated
audit, gap logging for Partial/Gap, then completion. Python copies the selected
retrieved clause's full original text into the evidence field.

Errors and incomplete workflows become `INVALID` predictions. Citation provenance
and successful tool execution do not establish judgment accuracy. Compare predicted
labels and selected clauses with the benchmark after execution.

In [6]:
evaluation_set = benchmark.iloc[:limit].copy()
evaluation_requirements = requirements[:limit]
print(f"Selected {len(evaluation_requirements)} requirements")

Selected 5 requirements


In [7]:
records = []
for index, (requirement_id, requirement_text) in enumerate(evaluation_requirements, 1):
    record = run_agent(
        requirement_id,
        requirement_text,
        vector_store,
        client=client,
        k=retrievals,
        max_steps=max_steps,
        verbose=verbose,
        reasoning=reasoning,
        agent_num_predict=agent_num_predict,
    )
    records.append(record)
    status = record.verdict.verdict if record.verdict else "FAILED"
    print(f"[{index}/{len(evaluation_requirements)}] {requirement_id}: {status}", flush=True)

[1/5] 1: Partial
[2/5] 2: Partial
[3/5] 3: Partial
[4/5] 4: Meets
[5/5] 5: Partial


In [8]:
rows, metrics, matrix = evaluate(evaluation_set, records, k=retrievals)

print("Evaluation mode: Agent")
print(f"Requirements evaluated: {metrics['n']}")
print(f"Verdict accuracy: {metrics['verdict_accuracy']:.1%}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Retrieval Hit@{retrievals}: {metrics['retrieval'][f'hit_at_{retrievals}']:.1%}")
print(f"Accepted verdict rate: {metrics['schema_valid_rate']:.1%}")
print(f"Failed audits: {sum(record.verdict is None for record in records)}")

display(matrix)
display(rows)

Evaluation mode: Agent
Requirements evaluated: 5
Verdict accuracy: 80.0%
Macro F1: 0.5079
Retrieval Hit@5: 100.0%
Accepted verdict rate: 100.0%
Failed audits: 0


model_verdict,Meets,Partial,Gap,INVALID
human_verdict,,,,
Meets,1,1,0,0
Partial,0,3,0,0
Gap,0,0,0,0


,requirement_id,swe_id,requirement_text,clause_id,label,rationale,pred_verdict,pred_clause,retrieved,hit_at_1,hit_at_k,citation_member,correct
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone...",Partial,4.1.2,"4.1.2,4.1.3,2.2.8,4.1.4,2.1.5.12",True,True,True,False
1,2,SWE-051,The software requirements shall be derived fro...,4.1.3,Partial,Requirements analysis traces to systems engine...,Partial,4.1.3,"4.1.2,4.1.3,2.1.8.2,3.1.13,4.1.4",False,True,True,True
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,...",Partial,4.1.5,"4.1.5,2.2.7,4.1.2,3.5.2,2.1.8.2",True,True,True,True
3,4,SWE-055,The system shall undergo requirements validati...,4.1.7,Meets,Validation activity confirms the software will...,Meets,4.1.7,"4.1.7,4.1.2,4.5.12,4.1.3,4.5.8",True,True,True,True
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi...",Partial,4.2.3,"4.2.3,4.3.2,3.1.6,4.1.2,4.2.4",True,True,True,True


## Inspect Incorrect Predictions, Failures, and Findings

In [9]:
display(rows.loc[~rows["correct"]])

for record in records:
    if record.error:
        print(f"{record.requirement_id}: FAILED — {record.error}")
    elif record.finding_id:
        print(f"{record.requirement_id}: {record.verdict.verdict} — finding {record.finding_id}")

,requirement_id,swe_id,requirement_text,clause_id,label,rationale,pred_verdict,pred_clause,retrieved,hit_at_1,hit_at_k,citation_member,correct
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone...",Partial,4.1.2,"4.1.2,4.1.3,2.2.8,4.1.4,2.1.5.12",True,True,True,False


1: Partial — finding 1:finding_1
2: Partial — finding 2:finding_1
3: Partial — finding 3:finding_1
5: Partial — finding 5:finding_1


## Measured Experiment Selected for Integration

The full source-evidence experiment used the settings above on all 50 development
requirements. It completed 50/50 audits with zero tool errors and matched 37/50
verdicts (74%, macro F1 0.7200). Expected clauses appeared in the top five for
49/50 requirements. All evidence fields matched their selected retrieved source.

Five requirements were incorrectly labeled Meets: **8, 13, 18, 20, and 28**.
These are experiment results, separate from the live results generated above.
The benchmark has been reused during development; it is not an independent holdout,
and successful tool use or valid citations do not prove compliance correctness.
See the [measured results and safeguards](../docs/agent.md#measured-source-evidence-experiment).
